# Activity I: Hacking Password

**Objective:** understand the concepts of hashing and salting through hands-on password-cracking and benchmarking exercises: brute-force attacks, rainbow-table attacks, and password analysis.

## Dependencies

This notebook requires:

- `hashlib` — Python standard library, provides MD5 / SHA-1 / SHA-256 / SHA-512 hashing.
- `bcrypt` — third-party library for password-specific hashing (`pip install bcrypt`).
- A password dictionary: [`10k-most-common.txt`](https://github.com/danielmiessler/SecLists/blob/master/Passwords/Common-Credentials/10k-most-common.txt) from SecLists, used for the dictionary attack in Exercise 1.

The cell below downloads the wordlist automatically if it isn't already present in the working directory.

In [1]:
!pip install bcrypt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.9 MB/s eta 0:00:0000:01


In [2]:
import hashlib
import bcrypt
import time
import itertools
import os
import urllib.request

In [3]:
WORDLIST_PATH = "10k-most-common.txt"
WORDLIST_URL = (
    "https://raw.githubusercontent.com/danielmiessler/SecLists/master/"
    "Passwords/Common-Credentials/10k-most-common.txt"
)

if not os.path.exists(WORDLIST_PATH):
    urllib.request.urlretrieve(WORDLIST_URL, WORDLIST_PATH)

with open(WORDLIST_PATH, encoding="utf-8", errors="ignore") as f:
    word_count = sum(1 for _ in f)

print(f"Wordlist ready: {WORDLIST_PATH} ({word_count} words)")

Wordlist ready: 10k-most-common.txt (10000 words)


## Exercise 01 — Dictionary Attack with Leetspeak Substitutions

**Objective:** understand how attackers use pre-built word lists (dictionaries) to crack hashes of common passwords.

**Scenario:** a SHA-1 hash was discovered in a compromised system:

```
d54cc1fe76f5186380a0939d2fc1723c44e8a5f7
```

The password is suspected to be a simple, common word — possibly with character substitutions (e.g. `o` → `0`, `l` → `1`, `i` → `1`) and/or case changes.

**Task:** read words from the dictionary, apply common substitutions and case variants, hash each candidate, and check it against the target hash.

In [4]:
TARGET_SHA1 = "d54cc1fe76f5186380a0939d2fc1723c44e8a5f7"

SUBSTITUTIONS = {
    "a": ["a", "A", "4"],
    "b": ["b", "B", "6", "8"],
    "c": ["c", "C"],
    "d": ["d", "D"],
    "e": ["e", "E", "3"],
    "f": ["f", "F"],
    "g": ["g", "G", "6", "9"],
    "h": ["h", "H"],
    "i": ["i", "I", "1"],
    "j": ["j", "J"],
    "k": ["k", "K"],
    "l": ["l", "L", "1"],
    "m": ["m", "M"],
    "n": ["n", "N"],
    "o": ["o", "O", "0"],
    "p": ["p", "P"],
    "q": ["q", "Q"],
    "r": ["r", "R"],
    "s": ["s", "S", "5"],
    "t": ["t", "T", "7"],
    "u": ["u", "U"],
    "v": ["v", "V"],
    "w": ["w", "W"],
    "x": ["x", "X"],
    "y": ["y", "Y"],
    "z": ["z", "Z", "2"],
    "0": ["0", "o", "O"],
    "1": ["1", "i", "I", "l"],
    "2": ["2", "z", "Z"],
    "3": ["3", "e", "E"],
    "4": ["4", "a", "A"],
    "5": ["5", "s", "S"],
    "6": ["6", "b", "G"],
    "7": ["7", "t", "T"],
    "8": ["8", "B"],
    "9": ["9", "g"],
}


def generate_variants(word):
    """Yield case variants and leetspeak-substitution variants of a word."""
    for case_word in {word, word.lower(), word.upper(), word.capitalize()}:
        options = [SUBSTITUTIONS.get(ch.lower(), [ch]) for ch in case_word]
        for combo in itertools.product(*options):
            yield "".join(combo)

In [5]:
def crack_sha1(target_hash, wordlist_path):
    with open(wordlist_path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            word = line.strip()
            if not word:
                continue
            for variant in generate_variants(word):
                if hashlib.sha1(variant.encode()).hexdigest() == target_hash:
                    return variant
    return None

In [6]:
start = time.time()
found_password = crack_sha1(TARGET_SHA1, WORDLIST_PATH)
elapsed = time.time() - start

print(f"Password found: {found_password!r}" if found_password else "Password NOT found")
print(f"Time taken: {elapsed:.2f} seconds")

Password found: 'ThaiLanD'
Time taken: 3.00 seconds


## Exercise 02 — Benchmarking Hash Algorithm Speed

**Objective:** understand why modern password-hashing algorithms like bcrypt are more secure than older ones like MD5 and SHA-1.

**Task:** measure how many hashes each algorithm can compute in a fixed amount of time. We test MD5, SHA-1, SHA-256, SHA-512, and bcrypt.

In [7]:
def benchmark_hashlib(algo_name, duration=3):
    count = 0
    start = time.time()
    while time.time() - start < duration:
        hashlib.new(algo_name, str(count).encode()).hexdigest()
        count += 1
    return count / (time.time() - start)

In [8]:
def benchmark_bcrypt(duration=1.5, rounds=12):
    count = 0
    start = time.time()
    while time.time() - start < duration:
        bcrypt.hashpw(str(count).encode(), bcrypt.gensalt(rounds=rounds))
        count += 1
    return count / (time.time() - start)

In [9]:
hash_rates = {}

for algo in ["md5", "sha1", "sha256", "sha512"]:
    hash_rates[algo] = benchmark_hashlib(algo)
    print(f"{algo}: {hash_rates[algo]:,.0f} hashes/sec")

md5: 650,014 hashes/sec
sha1: 710,090 hashes/sec
sha256: 612,118 hashes/sec
sha512: 450,298 hashes/sec


In [10]:
# bcrypt is intentionally slow, so a short duration is enough to get a stable rate.
# Increase `duration` for a more precise measurement.
hash_rates["bcrypt"] = benchmark_bcrypt(duration=1.5, rounds=12)
print(f"bcrypt (cost=12): {hash_rates['bcrypt']:,.1f} hashes/sec")

bcrypt (cost=12): 2.9 hashes/sec


## Exercise 03 — Estimating Brute-Force Time from Password Length

**Objective:** apply the performance measurements from Exercise 02 to understand the importance of password length and algorithm choice.

**Task:** estimate how long it would take an attacker to brute-force a password of a given length, assuming the password contains upper-case, lower-case, numbers, and symbols.

The search space for a password of length `L` over a character set of size `C` is `C^L`. Estimated crack time is:

$$\text{time} = \frac{C^L}{\text{hashes per second}}$$

In [11]:
CHARSET_SIZE = 94  # upper + lower + digits + symbols


def estimate_bruteforce_time(charset_size, length, hashes_per_sec):
    total_combinations = charset_size ** length
    return total_combinations / hashes_per_sec


def human_readable(seconds):
    years = seconds / (60 * 60 * 24 * 365)
    return f"{seconds:.2e} sec  (~{years:.2e} years)"

In [12]:
for algo, rate in hash_rates.items():
    print(f"\n--- {algo} ({rate:,.1f} hashes/sec) ---")
    for length in range(4, 13):
        t = estimate_bruteforce_time(CHARSET_SIZE, length, rate)
        print(f"length {length:2d}: {human_readable(t)}")


--- md5 (650,013.7 hashes/sec) ---
length  4: 1.20e+02 sec  (~3.81e-06 years)
length  5: 1.13e+04 sec  (~3.58e-04 years)
length  6: 1.06e+06 sec  (~3.37e-02 years)
length  7: 9.98e+07 sec  (~3.16e+00 years)
length  8: 9.38e+09 sec  (~2.97e+02 years)
length  9: 8.82e+11 sec  (~2.80e+04 years)
length 10: 8.29e+13 sec  (~2.63e+06 years)
length 11: 7.79e+15 sec  (~2.47e+08 years)
length 12: 7.32e+17 sec  (~2.32e+10 years)

--- sha1 (710,089.9 hashes/sec) ---
length  4: 1.10e+02 sec  (~3.49e-06 years)
length  5: 1.03e+04 sec  (~3.28e-04 years)
length  6: 9.72e+05 sec  (~3.08e-02 years)
length  7: 9.13e+07 sec  (~2.90e+00 years)
length  8: 8.58e+09 sec  (~2.72e+02 years)
length  9: 8.07e+11 sec  (~2.56e+04 years)
length 10: 7.59e+13 sec  (~2.41e+06 years)
length 11: 7.13e+15 sec  (~2.26e+08 years)
length 12: 6.70e+17 sec  (~2.13e+10 years)

--- sha256 (612,118.0 hashes/sec) ---
length  4: 1.28e+02 sec  (~4.04e-06 years)
length  5: 1.20e+04 sec  (~3.80e-04 years)
length  6: 1.13e+06 sec  (~3

**Discussion:** compare the length needed to exceed "1 year to brute-force" across algorithms above. Fast hashes (MD5/SHA-1/SHA-256) typically require noticeably longer passwords to reach that bar than bcrypt does, illustrating that **both** algorithm choice and password length matter for security.

## Exercise 04 — Is Brute-Forcing a Bcrypt Hash Practical?

**Question:** if a given hash value is from a bcrypt algorithm, is it practical to do a brute-force attack?

**Answer:** No, not for a reasonably long password. bcrypt is intentionally slow and has a tunable **cost factor**, so even with GPUs an attacker can only try a comparatively tiny number of guesses per second — see the `bcrypt` row measured in Exercise 02, which is thousands of times slower than MD5/SHA-1/SHA-256. Combined with the brute-force time estimates in Exercise 03, cracking an 8+ character password (mixed charset) via brute force against bcrypt is not feasible in a practical time frame. It only becomes practical if the password is very short or drawn from a small character set, or if a plain dictionary attack (rather than full brute force) succeeds against a very common password.

## Exercise 05 — Is a Rainbow-Table Attack Practical Against Bcrypt?

**Question:** if a given hash value is from a bcrypt algorithm, is it practical to perform a rainbow-table attack?

**Answer:** No. bcrypt automatically generates and embeds a unique, random **salt** in every hash it produces. Rainbow tables rely on precomputing hashes for *unsalted* inputs so a single table can attack any account sharing a password; since every bcrypt hash uses a different salt, an attacker would need a separate table per unique salt, which defeats the entire efficiency advantage of a rainbow table. Combined with bcrypt's slowness (which also makes precomputation itself expensive), rainbow-table attacks are not practical against bcrypt.

The cell below demonstrates this directly: hashing the *same* password twice produces two *different* bcrypt hashes, because each call generates a fresh random salt.

In [13]:
password = b"Chulalongkorn"

hash1 = bcrypt.hashpw(password, bcrypt.gensalt())
hash2 = bcrypt.hashpw(password, bcrypt.gensalt())

print("Hash 1:", hash1)
print("Hash 2:", hash2)
print("Same password, same hash?", hash1 == hash2)

Hash 1: b'$2b$12$GBdVxPhON.v3lIokActmyeFMsHEqDQAlfv6lCi30UYRSZijQWjeq.'
Hash 2: b'$2b$12$7SlZDDNkjwEwUDzY7DmcIe9cA1jBH0ecOfgPARy4Y7GuAXDxqRfWK'
Same password, same hash? False


## Exercise 06 — Secure Password Storage Design

**Task:** explain the design/strategy for securely storing a password in a database.

- **Proper hash function:** use a password-specific hashing algorithm such as bcrypt or Argon2 (Argon2id recommended), never a general-purpose fast hash like MD5/SHA-1/SHA-256 on its own.
- **Salting:** use a unique, randomly generated salt per user. bcrypt and Argon2 handle this automatically, so identical passwords never produce identical stored hashes, and precomputed/rainbow-table attacks become useless.
- **Cost factor:** tune the work factor (e.g. bcrypt cost 12–14) so hashing takes a noticeable fraction of a second on the server, balancing attacker cost against login latency, and increase it over time as hardware gets faster.
- **Never store plaintext passwords** anywhere — not in the database, logs, backups, or error messages.
- **Database security:** restrict access to the credentials table with least-privilege permissions, encrypt data at rest and in transit, use parameterized queries to prevent SQL injection, and monitor/log access to sensitive tables.
- **Defense in depth:** add login rate-limiting/account lockout, multi-factor authentication, and reject known-breached passwords (e.g. check against a list like `10k-most-common.txt` used in Exercise 01) at signup/change time.